In [ ]:
import sys
import os

notebook_path = os.getcwd() 
parent_dir = os.path.dirname(notebook_path)
project_root = os.path.dirname(parent_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from copy import deepcopy

In [3]:
import state_NN_models
import Filters
import utils
import Systems
from utils import losses, trainer, utils
from torch.utils.data import TensorDataset, DataLoader, random_split
from state_NN_models.StateBayesianKalmanNet import StateBayesianKalmanNet
from state_NN_models.StateKalmanNet import StateKalmanNet
from state_NN_models.StateKalmanNetWithKnownR import StateKalmanNetWithKnownR

In [4]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
print(f"Používané zařízení: {device}")

Používané zařízení: cpu


In [5]:
state_dim_2d = 2
obs_dim_2d = 2

F_base_2d = torch.tensor([[1.0, 1.0], 
                          [0.0, 1.0]])

svd_F = torch.linalg.svd(F_base_2d)
F_true_2d = F_base_2d / svd_F.S[0]
print(F_true_2d)

H_true_2d = torch.eye(obs_dim_2d)

Q_true_2d = torch.eye(state_dim_2d) * 0.5 # Šum procesu
R_true_2d = torch.eye(obs_dim_2d) * 0.1 # Šum měření

# Počáteční podmínky
Ex0_true_2d = torch.tensor([[1.0], [0.0]])
P0_true_2d = torch.eye(state_dim_2d) * 1.5
F_model_2d = F_true_2d
H_model_2d = H_true_2d
Q_model_2d = torch.eye(state_dim_2d) * 0.1
R_model_2d = R_true_2d
Ex0_model_2d = torch.tensor([[0.5], [0.5]])
P0_model_2d = torch.eye(state_dim_2d) * 1.0

print("\nInicializuji 2D Linear_Canonical systém (replikace autorů)...")
sys_true = Systems.DynamicSystem(
    state_dim=state_dim_2d, obs_dim=obs_dim_2d,
    Ex0=Ex0_true_2d, P0=P0_true_2d,
    Q=Q_true_2d, R=R_true_2d,
    F=F_true_2d, H=H_true_2d,
    device=device
)
# sys_model = sys_true
# sys_model = Systems.DynamicSystem(
#     state_dim=state_dim_2d, obs_dim=obs_dim_2d,
#     Ex0=Ex0_model_2d, P0=P0_model_2d,
#     Q=Q_model_2d, R=R_model_2d,
#     F=F_model_2d, H=H_model_2d,
#     device=device
# )

sys_model=sys_true
print("... 2D systém inicializován.")

tensor([[0.6180, 0.6180],
        [0.0000, 0.6180]])

Inicializuji 2D Linear_Canonical systém (replikace autorů)...
... 2D systém inicializován.


In [6]:
import os
import matplotlib.pyplot as plt

def save_plot_eps(filename, folder_name="LinearSystemNoMismatchGraphs"):
    """
    Uloží aktuální matplotlib graf jako vektorový .eps soubor.
    
    Parametry:
    filename (str): Název souboru (bez přípony .eps)
    folder_name (str): Název cílové složky
    """
    # Vytvoření složky, pokud neexistuje (na stejné adresářové úrovni)
    os.makedirs(folder_name, exist_ok=True)
    
    # Sestavení absolutní cesty k souboru
    filepath = os.path.join(folder_name, f"{filename}.eps")
    
    # Uložení grafu 
    # bbox_inches='tight' je klíčové - zabrání oříznutí legendy ležící mimo osu
    plt.savefig(filepath, format='eps', bbox_inches='tight')
    print(f"--> Graf byl úspěšně uložen jako vektor: {filepath}")

In [7]:
TRAIN_SEQ_LEN = 10      # Krátké sekvence pro stabilní trénink (TBPTT)
VALID_SEQ_LEN = 20      # Stejná délka pro konzistentní validaci
TEST_SEQ_LEN = 100      # Dlouhé sekvence pro testování generalizace

NUM_TRAIN_TRAJ = 500   # Hodně trénovacích příkladů
NUM_VALID_TRAJ = 200    # Dostatek pro spolehlivou validaci
NUM_TEST_TRAJ = 100     # Pro robustní vyhodnocení

BATCH_SIZE = 32         # Dobrý kompromis

x_train, y_train = utils.generate_data(sys_true, num_trajectories=NUM_TRAIN_TRAJ, seq_len=TRAIN_SEQ_LEN)
x_val, y_val = utils.generate_data(sys_true, num_trajectories=NUM_VALID_TRAJ, seq_len=VALID_SEQ_LEN)
x_test, y_test = utils.generate_data(sys_true, num_trajectories=1, seq_len=TEST_SEQ_LEN)

train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import os
import random
import csv
from datetime import datetime
import pandas as pd
from copy import deepcopy

model_config = {
    "hidden_size_multiplier": 10,
    "output_layer_multiplier": 4,
    "num_gru_layers": 1,
    "init_min_dropout": 0.3,
    "init_max_dropout": 0.5
}

train_config = {
    "total_train_iter": 1800,
    "learning_rate": 1e-4,
    "clip_grad": 10.0,
    "J_samples": 10,
    "validation_period": 20,
    "logging_period": 20,
    "warmup_iterations":0 # Trénuj prvních 400 iterací jen na MSE
}
# =================================================================================
# KROK 3: SPUŠTĚNÍ JEDNOHO TRÉNINKOVÉHO BĚHU NEBO NAČTENÍ HOTOVÉHO MODELU
# =================================================================================

print("="*80)
print("Připravuji Bayesian KalmanNet (BKN)...")
print(f"Parametry modelu: {model_config}")
print(f"Parametry tréninku: {train_config}")
print("="*80)

# Nastavení seedu pro reprodukovatelnost tohoto běhu
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Definice cesty ke složce a souboru
save_dir = "LinearSystemWeights"
os.makedirs(save_dir, exist_ok=True)
weights_path = os.path.join(save_dir, "best_bkn_model_weights-latest-test.pth")

# Vytvoření modelu (potřebujeme ho inicializovat v obou případech)
state_bkn_knet = StateBayesianKalmanNet(
    sys_model,
    device=device,
    weight_init=False,
    log_history=False,
    **model_config
).to(device)

# Podmínka pro načtení nebo trénink
if os.path.exists(weights_path):
    print(f"\nINFO: Uložené váhy nalezeny v '{weights_path}'.")
    print("Přeskakuji trénink a načítám hotový model...")
    
    # Načtení vah
    state_bkn_knet.load_state_dict(torch.load(weights_path, map_location=device))
    trained_model = state_bkn_knet
    
else:
    print("\nINFO: Váhy nebyly nalezeny. Spouštím plnohodnotný tréninkový běh...")
    
    # Spuštění tréninku
    results = trainer.training_session_trajectory_with_gaussian_nll_training_fcn(
        model=state_bkn_knet,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        **train_config
    )

    # `run_training_session` automaticky načte nejlepší model zpět,
    # takže `state_bkn_knet` nyní obsahuje váhy nejlepšího modelu.
    trained_model = results['final_model']
    
    # Uložení vah pro příští použití
    torch.save(trained_model.state_dict(), weights_path)
    print(f"\nINFO: Váhy modelu byly úspěšně uloženy do: {weights_path}")

    # Výpis metrik pouze pokud jsme trénovali
    print("\n" + "="*80)
    print("TRÉNINK DOKONČEN - FINÁLNÍ VÝSLEDKY Z NEJLEPŠÍHO MODELU")
    print("="*80)
    print(f"Nejlepší model byl nalezen v iteraci: {results['best_iter']}")
    print(f"Nejlepší dosažený validační ANEES: {results['best_val_anees']:.4f}")
    print("--- Metriky odpovídající tomuto nejlepšímu modelu ---")
    print(f"  MSE na validační sadě:       {results['best_val_mse']:.4f}")
    print(f"  NLL na validační sadě:       {results['best_val_nll']:.4f}")
    print("="*80)

# Prepnutí do evaluačního módu na konci bloku (klíčové pro BKN kvůli Dropoutu)
trained_model.eval()
# Výpočet celkového počtu trénovatelných parametrů
total_params = sum(p.numel() for p in trained_model.parameters() if p.requires_grad)
print(f"Celkový počet trénovatelných parametrů: {total_params}")
print("\nModel BKN je plně připraven v proměnné 'trained_model'.")

Připravuji Bayesian KalmanNet (BKN)...
Parametry modelu: {'hidden_size_multiplier': 10, 'output_layer_multiplier': 4, 'num_gru_layers': 1, 'init_min_dropout': 0.3, 'init_max_dropout': 0.5}
Parametry tréninku: {'total_train_iter': 1800, 'learning_rate': 0.0001, 'clip_grad': 10.0, 'J_samples': 10, 'validation_period': 20, 'logging_period': 20, 'warmup_iterations': 0}
loaded with input normalization

INFO: Uložené váhy nalezeny v 'LinearSystemWeights/best_bkn_model_weights-latest-test.pth'.
Přeskakuji trénink a načítám hotový model...
Celkový počet trénovatelných parametrů: 37478

Model BKN je plně připraven v proměnné 'trained_model'.


In [9]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import os
import random
import csv
from datetime import datetime
import pandas as pd
from copy import deepcopy
# Nastavení seedu pro reprodukovatelnost tohoto běhu
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

save_dir = "LinearSystemWeights"
weights_path = os.path.join(save_dir, "best_knet_model_weights-latest-test.pth")

# Vytvoření cílové složky (pokud ještě neexistuje, nic se nestane)
os.makedirs(save_dir, exist_ok=True)


state_knetR = StateKalmanNet(sys_model, device=device, hidden_size_multiplier=10,gru_hidden_dim_multiplier=4,returns_covariance=False,
                             log_history=False).to(device)

if os.path.exists(weights_path):
    print(f"INFO: Uložené váhy nalezeny v '{weights_path}'.")
    print("Přeskakuji trénink a načítám hotový model...")
    
    # Načtení vah (map_location zajistí kompatibilitu, pokud bys např. trénoval na GPU a načítal na CPU)
    state_knetR.load_state_dict(torch.load(weights_path, map_location=device))
    
else:
    print("INFO: Váhy nebyly nalezeny. Spouštím trénink modelu...")
    
    # Spuštění tréninku
    trainer.train_state_KalmanNet(
        model=state_knetR, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        device=device, 
        epochs=100, 
        lr=1e-4,
        weight_decay=1e-2,
        early_stopping_patience=30
    )
    
    # Po úspěšném tréninku váhy rovnou uložíme pro příští spuštění
    torch.save(state_knetR.state_dict(), weights_path)
    print(f"Trénink dokončen. Váhy byly bezpečně uloženy do '{weights_path}'.")

# Prepnutí do evaluačního módu (dobrá praxe před testováním)
state_knetR.eval()
# Výpočet celkového počtu trénovatelných parametrů
total_params = sum(p.numel() for p in state_knetR.parameters() if p.requires_grad)
print(f"Celkový počet trénovatelných parametrů: {total_params}")

DEBUG: Layer 'output_final_linear.0' initialized near zero (Start K=0).
INFO: Uložené váhy nalezeny v 'LinearSystemWeights/best_knet_model_weights-latest-test.pth'.
Přeskakuji trénink a načítám hotový model...
Celkový počet trénovatelných parametrů: 37460


In [12]:
import torch
import torch.nn.functional as F
import numpy as np
import time  # <-- MĚŘENÍ ČASU
from torch.utils.data import TensorDataset, DataLoader
import Filters
from utils import utils

# ==============================================================================
# STRIKTNÍ VYNUCENÍ CPU PRO CELÝ SKRIPT (LEVEL PLAYING FIELD)
# ==============================================================================
device = torch.device("cpu")
print(f"INFO: Skript běží striktně na zařízení: {device}")

# ==============================================================================
# 0. PŘEDPOKLADY - ZDE PŘIŘAĎTE VAŠE NATRÉNOVANÉ MODELY
# ==============================================================================
try:
    # Přesuneme modely natvrdo na CPU
    trained_model_bkn = trained_model.to(device)
    trained_model_knetR = state_knetR.to(device)
    
    # Přesun vnitřních parametrů systémového modelu u BKN/KNet
    if hasattr(trained_model_bkn, 'R'): trained_model_bkn.R = trained_model_bkn.R.to(device)
    if hasattr(trained_model_bkn, 'P0'): trained_model_bkn.P0 = trained_model_bkn.P0.to(device)
    if hasattr(trained_model_knetR, 'R'): trained_model_knetR.R = trained_model_knetR.R.to(device)
    if hasattr(trained_model_knetR, 'P0'): trained_model_knetR.P0 = trained_model_knetR.P0.to(device)
    trained_model_bkn.device = device
    trained_model_knetR.device = device
    
    print("INFO: Všechny natrénované modely nalezeny a přesunuty na CPU.")
except NameError:
    print("VAROVÁNÍ: Některé z proměnných `trained_model`, nebo `state_knetR` nebyly nalezeny.")

# ==============================================================================
# 1. KONFIGURACE TESTU
# ==============================================================================
TEST_SEQ_LEN = 1000
NUM_TEST_TRAJ = 1
J_SAMPLES_TEST = 50

# ==============================================================================
# 2. PŘÍPRAVA DAT
# ==============================================================================
print(f"\nGeneruji {NUM_TEST_TRAJ} testovacích trajektorií o délce {TEST_SEQ_LEN} pro evaluaci...")
# Přesun fyzikálního modelu na CPU pro generování dat a filtry
sys_true.F = sys_true.F.to(device)
sys_true.H = sys_true.H.to(device)
sys_true.Q = sys_true.Q.to(device)
sys_true.R = sys_true.R.to(device)
sys_true.Ex0 = sys_true.Ex0.to(device)
sys_true.P0 = sys_true.P0.to(device)

x_test, y_test = utils.generate_data(sys_true, num_trajectories=NUM_TEST_TRAJ, seq_len=TEST_SEQ_LEN)
x_test, y_test = x_test.to(device), y_test.to(device)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# ==============================================================================
# 3. INICIALIZACE KLASICKÝCH FILTRŮ PRO POROVNÁNÍ
# ==============================================================================
ekf_ideal = Filters.ExtendedKalmanFilter(sys_true)
ukf_ideal = Filters.UnscentedKalmanFilter(sys_true)
kf_ideal = Filters.KalmanFilter(sys_true)
ekf_ideal.device = device
ukf_ideal.device = device
kf_ideal.device = device

print("Všechny klasické model-based filtry (EKF, UKF, KF) inicializovány na CPU.")

# ==============================================================================
# 4. VYHODNOCOVACÍ SMYČKA
# ==============================================================================
all_x_true_cpu = []
all_x_hat_bkn_cpu, all_P_hat_bkn_cpu = [], []
all_x_hat_knetR_cpu, all_P_hat_knetR_cpu = [], []
all_x_hat_ekf_ideal_cpu, all_P_hat_ekf_ideal_cpu = [], []
all_x_hat_ukf_ideal_cpu, all_P_hat_ukf_ideal_cpu = [], []
all_x_hat_kf_ideal_cpu, all_P_hat_kf_ideal_cpu = [], []

print(f"\nVyhodnocuji modely na {NUM_TEST_TRAJ} testovacích trajektoriích (Vše CPU)...")

trained_model_bkn.eval() 
trained_model_knetR.eval()

# with torch.no_grad():
#     for i, (x_true_seq_batch, y_test_seq_batch) in enumerate(test_loader):
#         y_test_seq = y_test_seq_batch.squeeze(0)
#         x_true_seq = x_true_seq_batch.squeeze(0)
#         initial_state = x_true_seq[0, :].unsqueeze(0)
#         seq_len = y_test_seq.shape[0]
        
#         # --- A. Klasické filtry ---
#         kf_i_res = kf_ideal.process_sequence(y_test_seq, Ex0=sys_true.Ex0, P0=sys_true.P0)
#         full_x_hat_kf_i = kf_i_res['x_filtered']
#         full_P_hat_kf_i = kf_i_res['P_filtered']

#         ekf_i_res = ekf_ideal.process_sequence(y_test_seq, Ex0=sys_true.Ex0, P0=sys_true.P0)
#         full_x_hat_ekf_i = ekf_i_res['x_filtered']
#         full_P_hat_ekf_i = ekf_i_res['P_filtered']

#         ukf_i_res = ukf_ideal.process_sequence(y_test_seq, Ex0=sys_true.Ex0, P0=sys_true.P0)
#         full_x_hat_ukf_i = ukf_i_res['x_filtered']
#         full_P_hat_ukf_i = ukf_i_res['P_filtered']
        
#         # --- B. Bayesian KalmanNet ---
#         ensemble_trajectories = []
#         for j in range(J_SAMPLES_TEST):
#             trained_model_bkn.reset(batch_size=1, initial_state=initial_state)
#             current_x_hats = []
#             for t in range(1, TEST_SEQ_LEN):
#                 x_filtered_t, _ = trained_model_bkn.step(y_test_seq[t, :].unsqueeze(0))
#                 current_x_hats.append(x_filtered_t)
#             ensemble_trajectories.append(torch.cat(current_x_hats, dim=0))

#         ensemble = torch.stack(ensemble_trajectories, dim=0)
#         predictions_bkn = ensemble.mean(dim=0)
#         diff = ensemble - predictions_bkn.unsqueeze(0)
#         covariances_bkn = (diff.unsqueeze(-1) @ diff.unsqueeze(-2)).mean(dim=0)
        
#         full_x_hat_bkn = torch.cat([initial_state, predictions_bkn], dim=0)
#         full_P_hat_bkn = torch.cat([sys_true.P0.unsqueeze(0), covariances_bkn], dim=0)

#         # --- C. StateKalmanNetWithKnownR ---
#         trained_model_knetR.reset(batch_size=1, initial_state=initial_state)
#         knetR_preds_x, knetR_preds_P = [], []
#         for t in range(1, TEST_SEQ_LEN):
#             x_filtered_t, P_filtered_t = trained_model_knetR.step(y_test_seq[t, :].unsqueeze(0))
#             knetR_preds_x.append(x_filtered_t)
#             knetR_preds_P.append(P_filtered_t)
#         full_x_hat_knetR = torch.cat([initial_state, torch.cat(knetR_preds_x, dim=0)], dim=0)
        
#         processed_P_list = []
#         for p_tensor in knetR_preds_P:
#             while p_tensor.dim() < 2: p_tensor = p_tensor.unsqueeze(-1)
#             if p_tensor.dim() > 2 and p_tensor.shape[0] == 1: p_tensor = p_tensor.squeeze(0)
#             processed_P_list.append(p_tensor)

#         P_sequence_knetR = torch.stack(processed_P_list, dim=0)
#         P0_for_cat = sys_true.P0.clone()
#         while P0_for_cat.dim() < P_sequence_knetR.dim(): P0_for_cat = P0_for_cat.unsqueeze(0)
#         full_P_hat_knetR = torch.cat([P0_for_cat, P_sequence_knetR], dim=0)
        
#         # --- Ukládání do seznamů ---
#         all_x_true_cpu.append(x_true_seq)
        
#         all_x_hat_bkn_cpu.append(full_x_hat_bkn)
#         all_P_hat_bkn_cpu.append(full_P_hat_bkn)
#         all_x_hat_knetR_cpu.append(full_x_hat_knetR)
#         all_P_hat_knetR_cpu.append(full_P_hat_knetR)
#         all_x_hat_ekf_ideal_cpu.append(full_x_hat_ekf_i)
#         all_P_hat_ekf_ideal_cpu.append(full_P_hat_ekf_i)
#         all_x_hat_ukf_ideal_cpu.append(full_x_hat_ukf_i)
#         all_P_hat_ukf_ideal_cpu.append(full_P_hat_ukf_i)
#         all_x_hat_kf_ideal_cpu.append(full_x_hat_kf_i)
#         all_P_hat_kf_ideal_cpu.append(full_P_hat_kf_i)

#         print(f"Dokončena trajektorie {i + 1}/{NUM_TEST_TRAJ}...")

# # ==============================================================================
# # 5. FINÁLNÍ VÝPOČET A VÝPIS METRIK
# # ==============================================================================
# mse_bkn, anees_bkn = [], []; mse_knetR, anees_knetR = [], []
# mse_ekf_ideal, anees_ekf_ideal = [], []; mse_ukf_ideal, anees_ukf_ideal = [], []
# mse_kf_ideal, anees_kf_ideal = [], []

# print("\nPočítám finální metriky pro jednotlivé trajektorie...")

# with torch.no_grad():
#     for i in range(NUM_TEST_TRAJ):
#         x_true = all_x_true_cpu[i]
        
#         def get_metrics(x_hat, P_hat):
#             mse = F.mse_loss(x_hat[1:], x_true[1:]).item()
#             anees = utils.calculate_anees_vectorized(x_true.unsqueeze(0), x_hat.unsqueeze(0), P_hat.unsqueeze(0))
#             return mse, anees

#         mse, anees = get_metrics(all_x_hat_bkn_cpu[i], all_P_hat_bkn_cpu[i]); mse_bkn.append(mse); anees_bkn.append(anees)
#         mse, anees = get_metrics(all_x_hat_knetR_cpu[i], all_P_hat_knetR_cpu[i]); mse_knetR.append(mse); anees_knetR.append(anees)
#         mse, anees = get_metrics(all_x_hat_ekf_ideal_cpu[i], all_P_hat_ekf_ideal_cpu[i]); mse_ekf_ideal.append(mse); anees_ekf_ideal.append(anees)
#         mse, anees = get_metrics(all_x_hat_ukf_ideal_cpu[i], all_P_hat_ukf_ideal_cpu[i]); mse_ukf_ideal.append(mse); anees_ukf_ideal.append(anees)
#         mse, anees = get_metrics(all_x_hat_kf_ideal_cpu[i], all_P_hat_kf_ideal_cpu[i]); mse_kf_ideal.append(mse); anees_kf_ideal.append(anees)

# def avg(metric_list): return np.mean([m for m in metric_list if not np.isnan(m)])
# state_dim_for_nees = all_x_true_cpu[0].shape[1]

# print("\n" + "="*80)
# print(f"FINÁLNÍ VÝSLEDKY (průměr přes {NUM_TEST_TRAJ} běhů - PŘESNÁ ZNALOST MODELU)")
# print("="*80)
# print(f"{'Model':<35} | {'Průměrné MSE':<20} | {'Průměrný ANEES':<20}")
# print("-" * 80)
# print(f"{'--- Data-Driven Models ---':<35} | {'(nižší je lepší)':<20} | {'(bližší ' + str(float(state_dim_for_nees)) + ' je lepší)':<20}")
# print(f"{'Bayesian KNet (BKN)':<35} | {avg(mse_bkn):<20.4f} | {avg(anees_bkn):<20.4f}")
# print(f"{'KNet with Known R (KNetR)':<35} | {avg(mse_knetR):<20.4f} | {avg(anees_knetR):<20.4f}")
# print("-" * 80)
# print(f"{'--- Model-Based Filters ---':<35} | {'':<20} | {'':<20}")
# print(f"{'EKF (Ideální model)':<35} | {avg(mse_ekf_ideal):<20.4f} | {avg(anees_ekf_ideal):<20.4f}")
# print(f"{'UKF (Ideální model)':<35} | {avg(mse_ukf_ideal):<20.4f} | {avg(anees_ukf_ideal):<20.4f}")
# print(f"{'KF (Ideální model - OPTIMUM)':<35} | {avg(mse_kf_ideal):<20.4f} | {avg(anees_kf_ideal):<20.4f}")
# print("="*80)


# ==============================================================================
# 6. TEST VÝPOČETNÍ NÁROČNOSTI (INFERENCE TIME)
# ==============================================================================
print("\n" + "="*80)
print("SPUŠTĚNÍ TESTU VÝPOČETNÍ NÁROČNOSTI (STRIKTNĚ CPU, SEKVENČNÍ BĚH)")
print("="*80)

# NOVÉ: Specifická delší trajektorie pro robustnější měření času
INF_SEQ_LEN = 1000

# Generování 1 specifické trajektorie pro test rychlosti na CPU
x_inf, y_inf = utils.generate_data(sys_true, num_trajectories=1, seq_len=INF_SEQ_LEN)
y_seq_inf = y_inf[0]  # Je to už na CPU díky utils na začátku

inference_times = {}

with torch.no_grad():
    # --- 6.1 Měření Kalman Filter (Baseline) ---
    if not hasattr(kf_ideal, 'I'): kf_ideal.I = torch.eye(kf_ideal.state_dim, device=device)
    kf_ideal.reset(kf_ideal.Ex0, kf_ideal.P0)
    
    # Warmup
    for _ in range(50): kf_ideal.step(y_seq_inf[0])
    
    # Měření
    kf_ideal.reset(kf_ideal.Ex0, kf_ideal.P0)
    start_time = time.perf_counter()
    for t in range(INF_SEQ_LEN):
        kf_ideal.step(y_seq_inf[t])
    inference_times['KF'] = (time.perf_counter() - start_time) * 1000 / INF_SEQ_LEN

    # --- 6.2 Měření EKF ---
    ekf_ideal.reset(sys_true.Ex0, sys_true.P0)
    for _ in range(50): ekf_ideal.step(y_seq_inf[0])
    ekf_ideal.reset(sys_true.Ex0, sys_true.P0)
    start_time = time.perf_counter()
    for t in range(INF_SEQ_LEN):
        ekf_ideal.step(y_seq_inf[t])
    inference_times['EKF'] = (time.perf_counter() - start_time) * 1000 / INF_SEQ_LEN

    # --- 6.3 Měření UKF ---
    ukf_ideal.reset(sys_true.Ex0, sys_true.P0)
    for _ in range(50): ukf_ideal.step(y_seq_inf[0])
    ukf_ideal.reset(sys_true.Ex0, sys_true.P0)
    start_time = time.perf_counter()
    for t in range(INF_SEQ_LEN):
        ukf_ideal.step(y_seq_inf[t])
    inference_times['UKF'] = (time.perf_counter() - start_time) * 1000 / INF_SEQ_LEN

    # --- 6.4 Měření KalmanNet (KNetR) ---
    trained_model_knetR.eval()
    
    # Vypnutí logování a zpětného dopočtu matice P pro čistý test inference!
    if hasattr(trained_model_knetR, 'returns_covariance'): trained_model_knetR.returns_covariance = False
    if hasattr(trained_model_knetR, 'log_history'): trained_model_knetR.log_history = False
    
    trained_model_knetR.reset(batch_size=1)
    for _ in range(50): trained_model_knetR.step(y_seq_inf[0].unsqueeze(0))
    
    trained_model_knetR.reset(batch_size=1)
    start_time = time.perf_counter()
    for t in range(INF_SEQ_LEN):
        trained_model_knetR.step(y_seq_inf[t].unsqueeze(0))
    inference_times['KNet'] = (time.perf_counter() - start_time) * 1000 / INF_SEQ_LEN

    # --- 6.5 Měření Bayesian KalmanNet (BKN) - SEKVENČNÍ (SERIAL) BĚH ---
    trained_model_bkn.eval()
    
    # Vypnutí logování historie zisku
    if hasattr(trained_model_bkn, 'log_history'): trained_model_bkn.log_history = False
    
    # Warmup
    trained_model_bkn.reset(batch_size=1)
    for _ in range(50): trained_model_bkn.step(y_seq_inf[0].unsqueeze(0))
    
    start_time = time.perf_counter()
    
    # Striktně sekvenční zpracování J realizací (jak popsali autoři BKN)
    for j in range(J_SAMPLES_TEST):
        trained_model_bkn.reset(batch_size=1)
        for t in range(INF_SEQ_LEN):
            trained_model_bkn.step(y_seq_inf[t].unsqueeze(0))
            
    # Celkový čas zahrnuje sekvenční běh celkem J * INF_SEQ_LEN kroků.
    # Vydělením INF_SEQ_LEN získáme přesně průměrný čas potřebný na zpracování 
    # VŠECH J realizací pro JEDEN časový krok.
    inference_times['BKN (Serial)'] = (time.perf_counter() - start_time) * 1000 / INF_SEQ_LEN

print(f"{'Metoda':<20} | {'Průměrný čas [ms/krok]':<25}")
print("-" * 45)
for model_name, inf_time in inference_times.items():
    print(f"{model_name:<20} | {inf_time:<25.4f}")
print("="*80)

INFO: Skript běží striktně na zařízení: cpu
INFO: Všechny natrénované modely nalezeny a přesunuty na CPU.

Generuji 1 testovacích trajektorií o délce 1000 pro evaluaci...
Všechny klasické model-based filtry (EKF, UKF, KF) inicializovány na CPU.

Vyhodnocuji modely na 1 testovacích trajektoriích (Vše CPU)...

SPUŠTĚNÍ TESTU VÝPOČETNÍ NÁROČNOSTI (STRIKTNĚ CPU, SEKVENČNÍ BĚH)
Metoda               | Průměrný čas [ms/krok]   
---------------------------------------------
KF                   | 0.1067                   
EKF                  | 0.1255                   
UKF                  | 0.3059                   
KNet                 | 0.2830                   
BKN (Serial)         | 31.1799                  
